In [173]:
import warnings
warnings.filterwarnings('ignore')

In [174]:
import pandas as pd
import numpy as np
import pyreadr
import joblib
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import plotly.express as px

from tqdm import tqdm

from dash import Dash, html, dcc, dash_table, Input, Output, callback
import dash_bootstrap_components as dbc

In [175]:
pd.set_option('display.max_columns', 100)

In [176]:
dem_uncont = 34
rep_uncont = 31

In [177]:
seat_sims = pyreadr.read_r('model_output/tot_seats_sims.RDS')[None]
seat_sims = seat_sims.rename({None: 'seats'}, axis=1)
seat_sims['seats'] = seat_sims['seats'].map(lambda x: x + dem_uncont)
seat_sims['winner'] = seat_sims['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims

,seats,winner
0,54,Democrats
1,55,Democrats
2,48,Republicans
3,53,Democrats
4,48,Republicans
...,...,...
7995,51,Democrats
7996,51,Democrats
7997,53,Democrats
7998,51,Democrats


In [178]:
sim_counts = seat_sims.groupby(['seats']).count().reset_index().rename({'winner': 'count'}, axis=1)
n_sims = seat_sims.shape[0]
sim_counts['pct'] = sim_counts['count'] / n_sims * 100
sim_counts['winner'] = sim_counts['seats'].map(lambda x: 'Democrats' if x >= 51 else 'Republicans')
seat_sims = pd.merge(left=seat_sims.drop(['winner'], axis=1), right=sim_counts, on='seats', how='left')
def get_desc(winner, pct, seats):
    return f'{winner} wins {seats if seats >= 51 else (100-seats)} seats in {pct:.2f}% of simulations'
#seat_sims['desc'] = seat_sims[['winner', 'pct', 'seats']].apply(lambda x: get_desc(x['winner'], x['pct'], x['seats']), axis=1)
sim_counts.head()

,seats,count,pct,winner
0,36,1,0.0125,Republicans
1,38,1,0.0125,Republicans
2,40,2,0.0250,Republicans
3,41,3,0.0375,Republicans
4,42,12,0.1500,Republicans


In [179]:
np.unique(seat_sims['seats']).shape[0]

27

In [180]:
sims_hist = px.histogram(seat_sims, x='seats', nbins=np.unique(seat_sims['seats']).shape[0]*2, color='winner',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},
                         labels={'seats':'Seats won by Democrats', 'winner': 'Winner'}, template='plotly_white')
sims_hist.update_traces(showlegend=False)
sims_hist.add_vline(x=50.5, line_width=1, line_color='black', annotation_text='51 seats required for Dem majority', 
                    annotation_position='top right')
sims_hist

In [181]:
output_ts = pd.read_csv('model_output/output_over_time.csv')
output_ts.head()

,date,y,geo,type
0,2026-09-27,51.348375,US Senate,seats
1,2026-09-27,58.600000,US Senate,chance
2,2026-09-27,3.816193,US Senate,seats_sd
3,2026-09-27,38.771876,AL,y_pred
4,2026-09-27,3.537242,AL,y_pred_sd


In [182]:
chance_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'chance')]
chance_over_time['rep_chance'] = chance_over_time['y'].map(lambda x: 100 - x)
chance_over_time = chance_over_time.rename({'y': 'Democrats', 'rep_chance': 'Republicans'}, axis=1)
chance_time_ser = px.line(chance_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                         color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
chance_time_ser.update_traces(hovertemplate="%{y:.1f}%")
chance_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Win Probability (%)',
    title=dict(text="Senate Win Probability Over Time"),
    hovermode="x",
    showlegend=False
)
chance_time_ser

In [183]:
seats_over_time = output_ts[(output_ts['geo'] == 'US Senate') &
                             (output_ts['type'] == 'seats')]
seats_over_time['rep_seats'] = seats_over_time['y'].map(lambda x: 100 - x)
seats_over_time = seats_over_time.rename({'y': 'Democrats', 'rep_seats': 'Republicans'}, axis=1)
seats_time_ser = px.line(seats_over_time, x='date', y=['Democrats', 'Republicans'], template='plotly_white',
                        color_discrete_map={'Democrats': '#004b97', 'Republicans': '#c71e1d'},)
seats_time_ser.update_traces(hovertemplate="%{y:.1f}")
seats_time_ser.update_layout(
    xaxis=dict(range=[pd.to_datetime('2026-09-27'), pd.to_datetime('2026-11-10')]),
    yaxis=dict(range=[0, 100]),
    xaxis_title='Date',
    yaxis_title='Average Seats Over All Simulations',
    title=dict(text="Projected Seats Over Time"),
    hovermode="x",
    showlegend=False
)
seats_time_ser

In [184]:
joblib.dump(sims_hist, 'display_data/sims_histogram.pkl')
joblib.dump(chance_time_ser, 'display_data/chance_time_ser.pkl')
joblib.dump(seats_time_ser, 'display_data/seats_time_ser.pkl')

['display_data/seats_time_ser.pkl']

In [185]:
# posterior prediction
post = pyreadr.read_r('model_output/labeled_posterior.RDS')[None]
post_untransp = post.copy()

In [186]:
post = post.T
post.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,...,7950,7951,7952,7953,7954,7955,7956,7957,7958,7959,7960,7961,7962,7963,7964,7965,7966,7967,7968,7969,7970,7971,7972,7973,7974,7975,7976,7977,7978,7979,7980,7981,7982,7983,7984,7985,7986,7987,7988,7989,7990,7991,7992,7993,7994,7995,7996,7997,7998,7999
AL,-5.268144,-11.805867,-9.746631,-10.017617,-7.827593,-11.306543,-9.812221,-3.578281,-10.065102,-15.504777,0.996378,-11.105124,-8.885616,-9.538535,-10.239714,-7.633278,-7.852267,-10.382929,-7.976206,-5.435945,-10.714647,-7.056970,-5.045977,-7.277037,-6.756728,-12.619925,-12.073193,-6.023555,-11.075923,-11.242677,-8.632044,-9.807295,-9.521453,-6.055864,-14.142189,-13.123111,-5.924338,-8.963974,-6.647885,-10.717990,-8.602493,-6.837958,-11.924615,-8.774885,-10.924839,-11.483731,-10.988052,-13.049344,-6.005438,-9.548165,...,-9.711667,-9.067878,-7.872552,-13.158400,-5.117556,-9.783418,-10.887288,-5.770132,-10.242060,-6.283294,-14.290249,-14.291759,-8.550809,-5.226673,-15.575753,-5.166935,-14.526527,-1.078146,-10.007120,-6.915612,-9.130099,-7.805949,-2.724522,-19.336216,-1.371651,-11.914305,-9.552910,-11.572475,-5.841644,-10.566691,-8.817584,-10.853618,-8.972329,-6.046088,-14.792822,-10.376377,-8.996999,-16.820314,-7.997080,-8.103393,-8.969334,-12.188951,-8.659593,-8.541617,-10.200887,-11.065426,-13.072863,-7.944773,-10.725042,-10.238242
AK,0.577449,5.682794,-0.061640,5.597153,-1.038855,4.648610,4.948062,5.247059,-0.550143,-5.620264,10.919930,0.387581,-0.931210,6.612226,-0.752109,3.067812,5.665280,4.715424,-0.366231,2.866472,6.934933,4.226785,2.342083,5.181756,4.975950,1.344155,2.373319,3.649684,-0.906913,1.501154,3.392045,5.736735,7.064642,8.351502,2.797770,-0.284880,3.646771,0.044973,-0.998671,3.194298,-1.249165,3.715982,7.072467,4.058338,2.678009,-1.709987,0.997868,3.102899,2.093535,4.453639,...,3.978802,2.852418,10.042425,-0.265459,7.586097,2.665560,1.355003,3.665332,-1.446486,10.531991,2.399150,-0.913363,7.049888,3.347868,3.775105,1.573320,4.834331,2.195986,-0.532726,7.315155,1.784946,3.404058,8.263034,0.222646,3.877675,-2.183069,6.533453,4.819990,3.797906,0.823960,-3.129318,5.160389,-0.660769,6.005691,1.669705,3.085843,3.313513,7.182577,4.107249,4.017759,0.025641,4.246590,3.945092,0.219179,3.896920,1.034673,1.750931,0.705799,3.489865,1.716300
AR,-10.748693,-12.646419,-9.019748,-11.751716,-10.460854,-8.638540,-9.748333,-6.284979,-14.703445,-14.756928,-1.706572,-12.358385,-13.256242,-7.883201,-12.736891,-11.298996,-10.651011,-9.119210,-11.417299,-7.842025,-11.872434,-11.401652,-7.621513,-7.870760,-12.107071,-9.370765,-11.858047,-8.540635,-11.839942,-11.654844,-9.020645,-9.712967,-7.208430,-7.995659,-14.028807,-13.444203,-6.878300,-10.274031,-11.830108,-10.908979,-10.877049,-6.907936,-12.837895,-14.470399,-13.178051,-11.400408,-9.589392,-14.539110,-11.238344,-12.510181,...,-13.459047,-11.870576,-8.546234,-14.117271,-4.330415,-12.799263,-13.401499,-12.067450,-10.784229,-4.649156,-15.146336,-12.710272,-8.742396,-8.189419,-14.046706,-5.271294,-16.053766,-1.795986,-9.846470,-8.999560,-12.289599,-6.889764,-5.287853,-16.105304,-3.780639,-17.077548,-9.805602,-16.556188,-5.881914,-10.262495,-9.712047,-11.333881,-10.246716,-3.146072,-17.949830,-10.276420,-8.331742,-12.963809,-6.507370,-12.758009,-11.670842,-11.230399,-9.270283,-7.514630,-11.693759,-14.113090,-17.659474,-11.856270,-8.111448,-11.302522
CO,11.215880,12.002144,7.077561,13.924240,8.677254,12.372910,14.163348,13.063150,10.628647,6.958289,13.118850,13.612046,10.726901,15.985609,7.742798,15.471708,13.522344,11.614860,6.622344,5.892897,12.747038,18.512765,9.715774,16.245777,13.498217,11.295591,11.122592,19.870957,7.540325,13.343553,13.901864,10.283928,9.202597,10.294285,11.587782,5.627948,16.137117,11.017100,15.241662,9.020784,12.877058,11.848430,11.481157,13.412774,10.408897,11.545122,7.467889,10.931529,12.211282,9.349827,...,13.632581,8.452524,17.229665,10.062721,13.037556,6.731415,11.46739

In [187]:
sim_corr = post_untransp.corr()

In [188]:
sim_corr.head()

,AL,AK,AR,CO,DE,FL,GA,ID,IL,IA,KS,KY,LA,ME,MA,MI,MN,MS,MT,NE,NH,NJ,NM,NC,OH,OK,OR,RI,SC,SD,TN,TX,VA,WV,WY
AL,1.000000,0.516573,0.738739,0.490229,0.497070,0.698991,0.529682,0.735462,0.704521,0.739414,0.732202,0.540763,0.749402,0.732230,0.521266,0.729640,0.705216,0.742867,0.693598,0.720827,0.498910,0.506453,0.505069,0.702957,0.504766,0.756755,0.506974,0.487389,0.488077,0.512286,0.556164,0.694650,0.504801,0.739216,0.737274
AK,0.516573,1.000000,0.518614,0.522063,0.526168,0.507724,0.558374,0.526896,0.504636,0.522065,0.521129,0.568675,0.526084,0.523492,0.561320,0.513313,0.498988,0.518808,0.495279,0.517297,0.561142,0.540161,0.554078,0.498655,0.553815,0.522799,0.548093,0.549663,0.539132,0.543428,0.598445,0.506696,0.544564,0.521509,0.520730
AR,0.738739,0.518614,1.000000,0.487283,0.499839,0.703781,0.526900,0.737933,0.699195,0.741092,0.736842,0.537511,0.742057,0.733061,0.523147,0.728665,0.699286,0.726030,0.693421,0.720781,0.508876,0.500898,0.501600,0.704591,0.505065,0.744103,0.501882,0.494648,0.495298,0.514555,0.556801,0.699839,0.499127,0.735669,0.734963
CO,0.490229,0.522063,0.487283,1.000000,0.708931,0.651510,0.550601,0.497653,0.653938,0.502287,0.485550,0.549361,0.503910,0.485636,0.537835,0.487052,0.642447,0.493575,0.652316,0.475076,0.710164,0.715783,0.719744,0.646941,0.704598,0.499773,0.710414,0.700594,0.697713,0.528035,0.571408,0.654533,0.703957,0.496890,0.494077
DE,0.497070,0.526168,0.499839,0.708931,1.000000,0.665555,0.543260,0.504462,0.664570,0.507208,0.498315,0.547879,0.503227,0.502062,0.549066,0.501450,0.663770,0.499340,0.662891,0.487607,0.718599,0.725908,0.728648,0.661111,0.714945,0.515778,0.709991,0.711582,0.707688,0.519374,0.577975,0.658470,0.719022,0.499507,0.509408


In [189]:
post.shape

(35, 8000)

In [190]:
def get_tipping_point(sim):
    """
    :param sim: Series representing one posterior draw or "simulation"
    :type sim: pd.Series
    """
    seats_won_dem = np.sum(sim > 0)
    if seats_won_dem >= 51 - dem_uncont: # Democrats win House in this draw
        sim = sim.sort_values(ascending=True)
        won_seats = sim[sim > 0]
        seat_margin = seats_won_dem - (51 - dem_uncont)
    else: # Republicans win House in this draw
        sim = sim.sort_values(ascending=False)
        won_seats = sim[sim < 0]
        seat_margin = (100 - seats_won_dem - dem_uncont - rep_uncont) - (50 - rep_uncont)
    return won_seats.iloc[seat_margin - 1], won_seats.index[seat_margin - 1]

In [191]:
tipping_points = np.array([]) # tipping point for *each sim*

for i in tqdm(range(post.shape[1])):
    sim = post.iloc[:, i]
    _, tp_seat = get_tipping_point(sim)
    tipping_points = np.append(tipping_points, tp_seat)

tipping_points

100%|████████████████████████████████████████████████████████████████████████████| 8000/8000 [00:05<00:00, 1574.88it/s]


array(['NE', 'MI', 'MI', ..., 'AK', 'RI', 'AK'],
      shape=(8000,), dtype='<U32')

In [192]:
data = pd.read_csv('../../model_output/senate_predictions.csv')
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.271675,48.015860,1,96.031719,61.572967,3.503930,99.9500,4,54.787048,68.472288
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.364929,47.316437,1,94.632875,63.049134,3.504353,99.9875,5,56.211693,70.000493


In [193]:
data['tipping_point_prob'] = data['state_po'].map(lambda x: np.mean(tipping_points == x) * 100)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.0000
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.7500
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.0000
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.271675,48.015860,1,96.031719,61.572967,3.503930,99.9500,4,54.787048,68.472288,0.0875
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.364929,47.316437,1,94.632875,63.049134,3.504353,99.9875,5,56.211693,70.000493,0.1500


In [194]:
data.sort_values('tipping_point_prob', ascending=False).head(5)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob
15,15,Michigan,Abdul El-Sayed,Mike J. Rogers,False,False,D,MI,"EL-SAYED, ABDUL","ROGERS, MICHAEL J",4.719966e+07,15432313.28,6.263198e+07,75.360329,24.639671,-0.192456,49.278763,77.709102,2.333008,4.148895,12.968600,0.319413,32.445883,7404258,2673073,10077331,73.474395,26.525605,0.987401,-8.34254,0,0,49.461959,10.182798,45.679915,10.182798,East North Central,2026,0.0,0.00000,0.00000,2,Senate,5679.179195,10.182798,3.191050,-3.782043,7.957627,25.360329,0,50.720658,50.901941,3.539513,59.7375,16,44.118828,57.908472,11.0500
31,31,Texas,James Talarico,Ken Paxton,False,False,R,TX,"TALARICO, JAMES","PAXTON, WARREN KENNETH JR.",1.397766e+08,20997954.85,1.607746e+08,86.939504,13.060496,-5.916874,43.060966,48.286718,4.530331,31.642337,13.173480,0.180615,33.823334,24400697,4744808,29145505,83.720275,16.279725,0.987401,-8.34254,0,0,49.427997,12.983710,46.440125,12.983710,West South Central,2026,0.0,0.25284,-0.25284,6,Senate,7558.477337,12.983710,3.603292,-2.987872,-3.491209,36.939504,0,73.879008,51.383825,3.645696,64.9250,32,44.263135,58.646313,9.7875
13,13,Maine,Troy Jackson,Susan Collins,False,True,R,ME,"JACKSON, TROY DALE","COLLINS, SUSAN M.",2.397712e+07,15299978.61,3.927710e+07,61.046056,38.953944,3.820912,53.544306,94.249770,0.897865,1.689465,0.930085,0.342309,36.189622,526309,836050,1362359,38.632181,61.367819,0.987401,-8.34254,0,1,50.314182,7.880744,46.368587,7.880744,New England,2026,0.0,0.00000,0.00000,3,Senate,3726.620896,7.880744,2.807266,-3.945595,15.984364,11.046056,-1,22.092111,50.217583,3.579124,52.2375,14,43.197985,57.218437,9.5625
24,24,Ohio,Sherrod Brown,Jon Husted,False,True,R,OH,"BROWN, SHERROD","HUSTED, JON",5.786194e+07,9843192.51,6.770513e+07,85.461675,14.538325,-5.266554,44.343385,80.925356,1.708882,3.291326,11.486903,0.076606,31.487184,9001099,2798349,11799448,76.284069,23.715931,0.987401,-8.34254,0,1,48.921155,4.327021,46.259377,4.327021,East North Central,2026,0.0,0.00000,0.00000,2,Senate,7303.697897,4.327021,2.080149,-2.661777,-2.190568,35.461675,-1,70.923350,50.707285,3.471148,58.0250,25,43.818411,57.600003,8.4875
10,10,Kansas,Adam Hamilton,Roger Marshall,False,True,R,KS,"HAMILTON, ADAM","MARSHALL, ROGER W",1.652601e+07,2514153.97,1.904016e+07,86.795522,13.204478,-8.032578,41.794263,80.203805,2.235100,8.765322,5.263831,0.443291,35.603269,2124059,813821,2937880,72.299039,27.700961,0.987401,-8.34254,0,1,49.339039,5.188143,47.368125,5.188143,West North Central,2026,0.0,0.00000,0.00000,2,Senate,7533.462650,5.188143,2.277749,-1.970915,-7.722617,36.795522,-1,73.591044,49.422148,3.534535,43.0000,11,42.502174,56.443496,8.2375


In [195]:
def get_rating(dem_chance):
    if dem_chance > 100:
        raise ValueError('Invalid win chance.')
    if dem_chance > 95:
        return 'Safe D'
    elif dem_chance >= 90:
        return 'Very Likely D'
    elif dem_chance >= 75:
        return 'Likely D'
    elif dem_chance >= 65:
        return 'Lean D'
    elif dem_chance >= 60:
        return 'Tilt D'
    elif dem_chance >= 40:
        return 'Tossup'
    elif dem_chance >= 35:
        return 'Tilt R'
    elif dem_chance >= 25:
        return 'Lean R'
    elif dem_chance >= 10:
        return 'Likely R'
    elif dem_chance >= 5:
        return 'Very Likely R'
    else:
        return 'Safe R'

def get_matchup(dem_cand, rep_cand, dem_inc_any, rep_inc_any):
    indie_d = ['Dan Osborn', 'Brian Bengs', 'Todd Achilles', 'Seth Bodnar']
    indie_r = []
    
    if dem_cand in indie_d:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (I)'
        dem_color = '#792ba6'
    else:
        dem_lab = dem_cand + f'{'*' if dem_inc_any == True else ''}' + ' (D)'
        dem_color = '#366bbf'
    
    if rep_cand in indie_r:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (I)'
        rep_color = '#792ba6'
    else:
        rep_lab = rep_cand + f'{'*' if rep_inc_any == True else ''}' + ' (R)'
        rep_color = '#e63929'

    return f'<b style="color:{dem_color};">' + dem_lab + f'</b> vs <b style="color:{rep_color};">' + rep_lab + '</b>'

In [196]:
data['rating'] = data['chance'].map(lambda x: get_rating(x))
for party in ['dem', 'rep']:
    data[f'{party}_cand'] = data[f'{party}_cand'].map(lambda x: 'TBD' if x[:3] == 'TBD' else x)
data['matchup'] = data[['dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any']].apply(lambda x: get_matchup(x['dem_cand'], x['rep_cand'],
                                                                                                          x['dem_inc_any'], x['rep_inc_any']), axis=1)
data.head()

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.000000,0.0,0.000000,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.0000,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b..."
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.000000,0.0,0.000000,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.7500,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>..."
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.000000,0.0,0.000000,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.0000,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<..."
3,3,Colorado,John Hickenlooper,Mark Baisley,True,False,D,CO,"HICKENLOOPER, JOHN W.","BAISLEY, MARK",6665758.10,134935.30,6800693.40,98.015860,1.984140,5.964568,55.646542,73.317129,2.765032,16.968086,3.741915,0.400190,45.698090,4966936,806778,5773714,86.026707,13.973293,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,Mountain,2026,0.367879,0.0,0.367879,0,Senate,9607.108748,0.000000,0.000000,0.000000,20.271675,48.015860,1,96.031719,61.572967,3.503930,99.9500,4,54.787048,68.472288,0.0875,Safe D,"<b style=""color:#366bbf;"">John Hickenlooper* (..."
4,4,Delaware,Chris Coons,Michael Katz,True,False,D,DE,"COONS, CHRISTOPHER A.","KATZ, MICHAEL S DR.",2206861.15,60855.60,2267716.75,97.316437,2.683563,8.011194,57.479236,66.185281,3.007143,6.848324,21.107845,0.148276,35.764628,817817,172131,989948,82.612117,17.387883,0.987401,-8.34254,1,0,0.000000,0.000000,0.000000,0.000000,South Atlantic,2026,0.000000,0.0,0.000000,2,Senate,9470.488976,0.000000,0.000000,0.000000,24.364929,47.316437,1,94.632875,63.049134,3.504353,99.9875,5,56.211693,70.000493,0.1500,Safe D,"<b style=""color:#366bbf;"">Chris Coons* (D)</b>..."


In [197]:
data['projected_winner'] = data['chance'].map(lambda x: 'D' if x > 50 else 'R')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R


In [198]:
data['hold'] = data['projected_winner'] ==  data['current_party']
data['flip'] = data['hold'].map(lambda x: not x)
data['flip_indic'] = data['flip'].map(lambda x: 'Flip' if x else '')
#data['flip'] = data['flip'].map(lambda x: 'Yes' if x else 'No')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip


In [199]:
data['projected_2p_margin'] = data['y_pred'].map(lambda y_pred: f'D+{y_pred - (100-y_pred):.1f}' if y_pred > 50 else f'R+{(100-y_pred) - y_pred:.1f}')
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.1
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.3
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.8


In [200]:
data['rep_chance'] = data['chance'].map(lambda x: 100 - x)
data['rounded_dem_chance'] = data['chance'].map(lambda x: np.round(x, 1))
data['rounded_rep_chance'] = data['rep_chance'].map(lambda x: np.round(x, 1))
data['disp_dem_chance'] = data['rounded_dem_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data['disp_rep_chance'] = data['rounded_rep_chance'].map(lambda x: '>99%' if x > 99 else ('<1%' if x < 1 else str(x) + '%'))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.1,99.4375,0.6,99.4,<1%,>99%
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.3,17.3875,82.6,17.4,82.6%,17.4%
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.8,99.8250,0.2,99.8,<1%,>99%


In [201]:
data['swing_24_to_26'] = data['y_pred'].astype(float).map(lambda x: x - (100 - x)) - data['dem_2p_24'].astype(float).map(lambda x: x - (100 - x))
data.head(3)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.1,99.4375,0.6,99.4,<1%,>99%,12.817849
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.3,17.3875,82.6,17.4,82.6%,17.4%,20.037141
2,2,Arkansas,Hallie Shoffner,Tom Cotton,False,True,R,AR,"SHOFFNER, HALLIE","COTTON, THOMAS",3952722.44,3589862.47,7542584.91,52.405409,47.594591,-15.310522,34.330007,76.101134,1.188369,4.906064,14.703453,0.372273,25.731605,1670677,1340847,3011524,55.476131,44.523869,0.987401,-8.34254,0,1,39.806386,1.161279,52.528412,1.161279,West South Central,2026,0.0,0.0,0.0,5,Senate,2746.326849,1.161279,1.077627,12.722026,-22.278504,2.405409,-1,4.810817,39.622462,3.571219,0.1750,3,32.498465,46.573172,0.00,Safe R,"<b style=""color:#366bbf;"">Hallie Shoffner (D)<...",R,True,False,,R+20.8,99.8250,0.2,99.8,<1%,>99%,10.584911


In [202]:
data['disp_24_to_26_swing'] = data['swing_24_to_26'].map(lambda x: f'D+{x:.1f}' if x > 0 else f'R+{abs(x):.1f}')
data.head(2)

,...1,state,dem_cand,rep_cand,dem_inc_any,rep_inc_any,current_party,state_po,fec_dem_fuzzymatch,fec_rep_fuzzymatch,dem_funds,rep_funds,2p_funds,dem_funds_2p_pct,rep_funds_2p_pct,pvi,dem_2p_24,cvap_white_pct,cvap_aapi_pct,cvap_hisp_pct,cvap_black_pct,cvap_natam_pct,college,urban,rural,total,urban_pct,rural_pct,polarization,generic_ballot_avg,dem_inc_dummy,rep_inc_dummy,dem_poll_avg,dem_effn,rep_poll_avg,rep_effn,census_region,year,dem_scandal_score,rep_scandal_score,net_scandal_score,demo_cluster,chamber,dem_funds_2p_pct_sqrd,effn,sqrt_effn,poll_margin,baseline,dem_funds_2p_pct_offset,inc_dummy,funds_pct_margin,y_pred,y_pred_sd,chance,index,ci_low,ci_hi,tipping_point_prob,rating,matchup,projected_winner,hold,flip,flip_indic,projected_2p_margin,rep_chance,rounded_dem_chance,rounded_rep_chance,disp_dem_chance,disp_rep_chance,swing_24_to_26,disp_24_to_26_swing
0,0,Alabama,Everett Weiss,Barry Moore,False,False,R,AL,"WESS, EVERETT W","MOORE, FELIX BARRY",57926.93,2857769.50,2915696.43,1.986727,98.013273,-14.813221,34.559388,67.988934,1.025329,2.777706,25.742051,0.294693,28.369787,2900880,2123399,5024279,57.737240,42.262760,0.987401,-8.34254,0,0,38.246192,1.819209,49.145126,1.819209,East South Central,2026,0.0,0.0,0.0,5,Senate,3.947084,1.819209,1.348780,10.898934,-21.283902,-48.013273,0,-96.026546,40.968313,3.547290,0.5625,1,34.117948,48.089575,0.00,Safe R,"<b style=""color:#366bbf;"">Everett Weiss (D)</b...",R,True,False,,R+18.1,99.4375,0.6,99.4,<1%,>99%,12.817849,D+12.8
1,1,Alaska,Mary Peltola,Dan S. Sullivan,False,True,R,AK,"PELTOLA, MARY","SULLIVAN, DAN",32756766.41,5707701.05,38464467.46,85.161107,14.838893,-6.455302,43.153437,64.463948,6.481397,6.051383,3.077471,12.640423,31.669629,475967,257424,733391,64.899487,35.100513,0.987401,-8.34254,0,1,50.368983,3.275074,48.179944,3.275074,Pacific,2026,0.0,0.0,0.0,5,Senate,7252.414198,3.275074,1.809716,-2.189039,-4.568065,35.161107,-1,70.322215,53.172008,3.398757,82.6125,2,46.546373,59.965406,5.75,Likely D,"<b style=""color:#366bbf;"">Mary Peltola (D)</b>...",D,False,True,Flip,D+6.3,17.3875,82.6,17.4,82.6%,17.4%,20.037141,D+20.0


In [203]:
tab_data = data[['state', 'dem_cand', 'rep_cand', 'rating', 'disp_dem_chance', 'disp_rep_chance', 'projected_2p_margin', 'disp_24_to_26_swing',
                 'tipping_point_prob']]
for party in ['dem', 'rep']:
    tab_data[f'disp_{party}_chance'] = tab_data[f'disp_{party}_chance'].map(lambda x: f'<p style="color:{'blue' if party == 'dem' else 'red'};">{x}</p>')
    tab_data[f'{party}_cand'] = tab_data[f'{party}_cand'].map(lambda x: f'{x} (Ind)' if x in ['Dan Osborn', 'Brian Bengs', 
                                                                                              'Todd Achilles', 'Seth Bodnar'] else x)
tab_data = tab_data.rename({
    'state': 'State',
    'dem_cand': 'Democrat',
    'rep_cand': 'Republican',
    'rating': 'Rating',
    'disp_dem_chance': 'Dem Chance',
    'disp_rep_chance': 'Rep Chance',
    'projected_2p_margin': 'Projected Margin',
    'tipping_point_prob': 'Tipping Point Chance',
    'disp_24_to_26_swing': 'Swing from 2024 Pres'
}, axis=1)
tab_data.head()

,State,Democrat,Republican,Rating,Dem Chance,Rep Chance,Projected Margin,Swing from 2024 Pres,Tipping Point Chance
0,Alabama,Everett Weiss,Barry Moore,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+18.1,D+12.8,0.0000
1,Alaska,Mary Peltola,Dan S. Sullivan,Likely D,"<p style=""color:blue;"">82.6%</p>","<p style=""color:red;"">17.4%</p>",D+6.3,D+20.0,5.7500
2,Arkansas,Hallie Shoffner,Tom Cotton,Safe R,"<p style=""color:blue;""><1%</p>","<p style=""color:red;"">>99%</p>",R+20.8,D+10.6,0.0000
3,Colorado,John Hickenlooper,Mark Baisley,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+23.1,D+11.9,0.0875
4,Delaware,Chris Coons,Michael Katz,Safe D,"<p style=""color:blue;"">>99%</p>","<p style=""color:red;""><1%</p>",D+26.1,D+11.1,0.1500


In [204]:
data['rep_pred'] = data['y_pred'].map(lambda x: 100 - x)
data['projected_2p_margin_number'] = data['rep_pred'] - data['y_pred']
data['proj_seat_lean'] = data['projected_2p_margin_number'] - data['generic_ballot_avg']
sv_bias = float(data.sort_values('tipping_point_prob', ascending=False).reset_index().loc[0, 'proj_seat_lean']) # Seats-votes bias, + = R, - = D
sv_bias

6.538658106202197

In [205]:
topline_stats = pd.read_csv('display_data/topline_stats.csv')
if 'sv_bias' in topline_stats['vars'].values:
    topline_stats = topline_stats[topline_stats['vars'] != 'sv_bias']
topline_stats = pd.concat([topline_stats, pd.DataFrame({'vars': ['sv_bias'], 'x': sv_bias})], axis=0)
topline_stats.to_csv('display_data/topline_stats.csv')
topline_stats

,vars,x
0,means_seats_tot,51.988750
1,chamber_win_chance,62.825000
0,sv_bias,6.538658


In [206]:
mean_seats_tot = topline_stats[topline_stats['vars'] == 'means_seats_tot']['x'].values[0]
chamber_win_chance = topline_stats[topline_stats['vars'] == 'chamber_win_chance']['x'].values[0]

In [207]:
chances = topline_stats[topline_stats['vars'] == 'chamber_win_chance'].set_index(['vars']).T
chances['Republicans'] = chances['chamber_win_chance'].map(lambda x: 100 - x)
chances = chances.rename({'chamber_win_chance': 'Democrats'}, axis=1).T.reset_index()
chances = chances.rename({'x': 'Win Probability'}, axis=1)
seats = topline_stats[topline_stats['vars'] == 'means_seats_tot'].set_index(['vars']).T
seats['Republicans'] = seats['means_seats_tot'].map(lambda x: 100 - x)
seats = seats.rename({'means_seats_tot': 'Democrats'}, axis=1).T.reset_index().rename({'x': 'Seat Share'}, axis=1)
summary_stats = pd.merge(left=chances, right=seats, on='vars', how='inner').rename({'vars': 'Party'}, axis=1)
summary_stats.to_csv('display_data/summary_stats.csv')
summary_stats

,Party,Win Probability,Seat Share
0,Democrats,62.825,51.98875
1,Republicans,37.175,48.01125


In [208]:
incl_cols = ['state', 'dem_cand', 'rep_cand', 'dem_inc_any', 'rep_inc_any', 'rating', 'chance', 'disp_dem_chance',
            'disp_rep_chance', 'projected_2p_margin', 'y_pred', 'flip_indic', 'matchup', 'swing_24_to_26', 'disp_24_to_26_swing']
disp_data = data[incl_cols]

In [209]:
rating_ts = output_ts[output_ts['type'] == 'chance']
rating_ts['rating'] = rating_ts['y'].map(get_rating)
rating_ts['date'] = pd.to_datetime(rating_ts['date'])
rating_ts.head()

,date,y,geo,type,rating
1,2026-09-27,58.6000,US Senate,chance,Tossup
5,2026-09-27,0.0500,AL,chance,Safe R
10,2026-09-27,79.8875,AK,chance,Likely D
15,2026-09-27,0.1250,AR,chance,Safe R
20,2026-09-27,99.9625,CO,chance,Safe D


In [210]:
## Alert when there's a rating change
today_date = rating_ts['date'].max()
yesterday = rating_ts[rating_ts['date'] != today_date]['date'].max()

today_ratings = rating_ts[rating_ts['date'] == today_date]
yester_ratings = rating_ts[rating_ts['date'] == yesterday]
compare_df = pd.merge(left=today_ratings, right=yester_ratings, on='geo', how='left', suffixes=['_t', '_y'])
compare_df.head()

,date_t,y_t,geo,type_t,rating_t,date_y,y_y,type_y,rating_y
0,2026-11-01,62.8250,US Senate,chance,Tilt D,2026-10-31,62.2000,chance,Tilt D
1,2026-11-01,0.5625,AL,chance,Safe R,2026-10-31,0.5750,chance,Safe R
2,2026-11-01,82.6125,AK,chance,Likely D,2026-10-31,82.7625,chance,Likely D
3,2026-11-01,0.1750,AR,chance,Safe R,2026-10-31,0.1750,chance,Safe R
4,2026-11-01,99.9500,CO,chance,Safe D,2026-10-31,99.9500,chance,Safe D


In [211]:
## This is the cell that actually alerts when there's a rating change
if any(compare_df['rating_t'] != compare_df['rating_y']):
    changes_df = compare_df[compare_df['rating_t'] != compare_df['rating_y']][['geo', 'rating_y', 'rating_t']]
    print(changes_df)
else:
    print("No rating changes today!")

No rating changes today!


In [212]:
for i in range(changes_df.shape[0]):
    seat = changes_df.iloc[i, :]['geo']
    rating_y = changes_df.iloc[i, :]['rating_y']
    rating_t = changes_df.iloc[i, :]['rating_t']
    print(f'{seat}: {rating_y} --> {rating_t}')

KS: Tilt R --> Tossup


In [213]:
if any(compare_df['rating_t'] != compare_df['rating_y']):
    print(changes_df.shape)

In [224]:
## Mini-widget: what is the chance of Dems/Reps winning Senate if X state is won by Dems/Reps?
def det_winner(post, state, dem_winner=True):
    '''
    :param post: Posterior simulation matrix.
    :param state: Congressional district.
    :param dem_winner: Whether or not we set Dems or Reps to win `state`. True if Dem win is fixed, False if Rep win is fixed.
    '''
    winners = []
    seats = []
    post_ut = post.T
    if dem_winner:
        post_ut = post_ut[post_ut[state] > 0]
    else:
        post_ut = post_ut[post_ut[state] < 0]
    post_case = post_ut.T
    for i in range(post_case.shape[0]):
        sim = post_case.iloc[:, i]
        dem_seats = np.sum(sim > 0)
        if dem_seats >= 51 - dem_uncont: # Democrats win House in this draw
            winners.append(1)
        else: # Republicans win House in this draw
            winners.append(0)
        seats.append(dem_seats)
    dem_win_chance = np.mean(np.array(winners)) * 100
    dem_mean_seats = np.mean(np.array(seats)) + dem_uncont
    return dem_win_chance, dem_mean_seats

st = 'TX'
dem_winner = True
cond_win_chance, cond_seats = det_winner(post, st, dem_winner)
print(f'If Dems {'win' if dem_winner else 'lose'} {st}-SEN, they have a {cond_win_chance:.1f}% chance of winning the chamber overall, with mean projected seats of {cond_seats:.1f}')

If Dems win TX-SEN, they have a 82.9% chance of winning the chamber overall, with mean projected seats of 53.3


In [215]:
disp_data.to_csv('display_data/choropleth_display_data.csv')
tab_data.to_csv('display_data/table_display_data.csv')
data.to_csv('display_data/all_data.csv')